# PyRosetta Scoring Functions

![PyRosetta Scoring Functions](https://proto-bio.github.io/proto-assets/images/tool/pyrosetta/hero.png)

This notebook demonstrates the five PyRosetta operations wrapped in `proto_tools`: Spatial Aggregation Propensity (SAP), Solvent Accessible Surface Area (SASA), Rosetta energy scoring, FastRelax (returns a relaxed `Structure`), and interface analysis for two-chain complexes. Each tool takes one or more protein structures and returns physics-based metrics with per-residue breakdowns where applicable.

**License:** PyRosetta requires acceptance of the [Rosetta Software License](https://www.rosettacommons.org/software/license-and-download). Free for academic and non-commercial use; commercial users must obtain a separate license.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyrosetta")
display_overview("pyrosetta")
display_docs_section("pyrosetta", "Background")

## Available tools

In [ ]:
display_available_tools("pyrosetta")

### `run_pyrosetta_relax`

FastRelax minimizes a structure under the chosen score function and returns the relaxed coordinates as a `Structure`. Use this when downstream filters or scorers need to operate on a relaxed pose (e.g. cofolding filter pipelines for binder design). The relaxed `Structure` chains directly into the other PyRosetta scorers or into geometric `Structure` methods.


In [ ]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    PyRosettaRelaxInput,
    run_pyrosetta_energy,
    run_pyrosetta_relax,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax import example_input as relax_example_input


In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_relax")

relax_inputs = relax_example_input()


In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_relax")

# Single FastRelax cycle to match Germinal's `-relax:default_repeats 1` and keep the example fast
relax_config = PyRosettaRelaxConfig(scorefxn="ref2015", relax_cycles=1)


In [ ]:
# Run FastRelax
relax_result = run_pyrosetta_relax(relax_inputs, relax_config)


> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_relax")

r = relax_result.results[0]
print(f"Total score (relaxed): {r.total_score:.1f} REU")
print(f"Cycles applied:        {r.relax.relax_cycles}")
print(f"Output chains:         {r.relax.relaxed_structure.get_chain_ids()}")

# The relaxed Structure is ready to chain into another scorer.
relaxed_structure = r.relax.relaxed_structure
relaxed_structure.visualize(color_by="chain")


In [ ]:
# Chain the relaxed Structure into pyrosetta-energy. Default config = no further relax.
chained = run_pyrosetta_energy(
    PyRosettaEnergyInput(inputs=[relaxed_structure]),
)
print(f"Re-scored relaxed pose: {chained.results[0].total_energy:.1f} REU")


### `run_pyrosetta_sap`

SAP (Spatial Aggregation Propensity) quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to aggregation — a major concern in therapeutic protein development. `run_pyrosetta_sap` loads each input structure into a Rosetta Pose and scores it with Rosetta's `core.pack.guidance_scoreterms.sap` module, returning a single scalar SAP score per structure. Higher SAP scores indicate greater aggregation risk.

In [ ]:
from proto_tools import (
    PyRosettaSAPConfig,
    PyRosettaSAPInput,
    run_pyrosetta_sap,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sap import example_input as sap_example_input

In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sap")

# Use the tool's canonical example input PDB.
sap_inputs = sap_example_input()

# Let's take a look at what this looks like
sap_inputs.inputs[0].structure.visualize(color_by="chain")

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sap")

# PyRosettaSAPConfig has no tool-specific parameters; all defaults are used
sap_config = PyRosettaSAPConfig()

In [ ]:
# Run the SAP scoring function
sap_result = run_pyrosetta_sap(sap_inputs, sap_config)

In [ ]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_sap")

for r in sap_result.results:
    print(f"SAP score: {r.sap_score:.2f}")

### `run_pyrosetta_sasa`

SASA (Solvent Accessible Surface Area) measures the total protein surface accessible to a spherical solvent probe (1.4 A by default, matching water). Buried residues contribute to the hydrophobic core, while exposed residues interact with solvent and binding partners. `run_pyrosetta_sasa` returns both the total SASA and a per-residue breakdown, which is useful for identifying surface-exposed sites, characterizing the hydrophobic core, or feeding into downstream feature engineering.

In [ ]:
from proto_tools import (
    PyRosettaSASAConfig,
    PyRosettaSASAInput,
    run_pyrosetta_sasa,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sasa import example_input as sasa_example_input

In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sasa")

sasa_inputs = sasa_example_input()

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sasa")

# 1.4 A is the standard water probe radius
sasa_config = PyRosettaSASAConfig(probe_radius=1.4)

In [ ]:
# Run the SASA computation
sasa_result = run_pyrosetta_sasa(sasa_inputs, sasa_config)

In [ ]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_sasa")

# Total SASA plus the five most solvent-exposed residues
r = sasa_result.results[0]
print(f"Total SASA: {r.total_sasa:.1f} A^2")
print(f"Residues:   {len(r.per_residue)}")

top_exposed = sorted(r.per_residue, key=lambda res: res.sasa, reverse=True)[:5]
print("\nTop 5 most exposed residues:")
for res in top_exposed:
    print(f"  {res.chain_id}:{res.residue_name}{res.residue_index} — {res.sasa:.1f} A^2")

### `run_pyrosetta_energy`

Rosetta energy scoring evaluates a protein structure with a physics-based energy function that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion terms. The score is reported in Rosetta Energy Units (REU); lower (more negative) is more favorable. By default, `run_pyrosetta_energy` scores the input structure as-given; setting `pre_relax_structures=True` on the config opts into the [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) preprocess (composing `pyrosetta-relax` under the hood) to resolve clashes and backbone strain first. Returns the total energy, a per-term breakdown, and per-residue totals. The default score function is `ref2015`, Rosetta's current production all-atom energy function.


In [ ]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    run_pyrosetta_energy,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_energy import example_input as energy_example_input


In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_energy")

energy_inputs = energy_example_input()

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_energy")

# Use ref2015 with the FastRelax preprocess (1 cycle to keep the example fast)
energy_config = PyRosettaEnergyConfig(
    scorefxn="ref2015",
    pre_relax_structures=True,
    relax_config=PyRosettaRelaxConfig(relax_cycles=1),
)


In [ ]:
# Run the energy scoring function
energy_result = run_pyrosetta_energy(energy_inputs, energy_config)

In [ ]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_energy")

r = energy_result.results[0]
print(f"Total energy: {r.total_energy:.1f} REU")
print("\nTop contributing energy terms:")
for term, value in sorted(r.energy_terms.items(), key=lambda kv: kv[1])[:6]:
    print(f"  {term:>12s}: {value:.2f}")


### `run_pyrosetta_interface_analyzer`

`InterfaceAnalyzerMover` + `LayerSelector` produce a suite of interface-quality metrics for complexes: shape complementarity, interface H-bond count, binding ΔG, buried SASA, interface packing, and interface/surface hydrophobicity. The interface is identified by the `target_chains` / `binder_chain` fields on each `InterfaceStructureInput` (alongside the structure itself) — chain labels are validated against the structure at construction time.

In [ ]:
from proto_tools import (
    PyRosettaInterfaceAnalyzerConfig,
    PyRosettaInterfaceAnalyzerInput,
    run_pyrosetta_interface_analyzer,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_interface_analyzer import (
    example_input as interface_example_input,
)


In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_interface_analyzer")

interface_inputs = interface_example_input()


In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_interface_analyzer")

# Pre-relax the cofolded pose so the interface dG is physically meaningful (negative).
interface_config = PyRosettaInterfaceAnalyzerConfig(pre_relax_structures=True)


In [ ]:
# Run the interface analyzer
interface_result = run_pyrosetta_interface_analyzer(interface_inputs, interface_config)


In [ ]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_interface_analyzer")

m = interface_result.results[0]
print(f"interface_sc:                 {m.interface_sc:.3f}")
print(f"interface_hbonds:             {m.interface_hbonds}")
print(f"interface_dG:                 {m.interface_dG:.2f} REU")
print(f"interface_dSASA:              {m.interface_dSASA:.1f} Å²")
print(f"interface_packstat:           {m.interface_packstat:.3f}")
print(f"interface_hydrophobicity:     {m.interface_hydrophobicity:.1f} %")
print(f"surface_hydrophobicity:       {m.surface_hydrophobicity:.3f}")


## Export Results

Each output model supports export to common file formats for downstream analysis. SASA results export well to CSV because of the per-residue table; energy results export to JSON to preserve the nested per-term breakdown.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sap_result.export("pyrosetta_sap", export_path=output_dir, file_format="json")
sasa_result.export("pyrosetta_sasa", export_path=output_dir, file_format="csv")
energy_result.export("pyrosetta_energy", export_path=output_dir, file_format="json")
interface_result.export("pyrosetta_interface", export_path=output_dir, file_format="csv")

for name in (
    "pyrosetta_sap.json",
    "pyrosetta_sasa.csv",
    "pyrosetta_energy.json",
    "pyrosetta_interface.csv",
):
    print(f"Exported {output_dir / name}")